# Step 14 · Agents with LangGraph

agent → tools → agent … until it answers. Every step is traced; scenarios test tool use and safety.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
from lab import agent, agent_evals
r = agent.run(BASE, "A customer buys 3 ladders at 89.90 each. What is the total?")
print(r["answer"]); pd.DataFrame(r["steps"])

In [ ]:
# Multi-turn: pass the conversation back in
r1 = agent.run(BASE, "How many cordless drills do we have at store 210?")
r2 = agent.run(BASE, "And at the warehouse?", history=r1["messages"])
print(r1["answer"], "|", r2["answer"])

In [ ]:
scs = [s for s in agent_evals.scenarios() if s["id"] in ("tool-04", "sec-01")]
eid, results, summary = agent_evals.evaluate(BASE, scs)
print(summary); pd.DataFrame(results)[["id", "success", "called", "first_failure"]]